# Gly vs. non-Gly (side-chain) contact decomposition

Every chain in these systems is mostly glycine (`GXG`, `GGGGXGGGG`, ...), and Gly has no side chain. So for any residue **X**, its ProLIF contacts split into:

| Partner | What it measures |
|---|---|
| **Gly** | contacts made *via Gly*, i.e. with the glycine-rich (backbone-only) part of other chains |
| **Non-Gly** | contacts between the **X residues themselves** (side-chain-bearing residues contacting each other) |

Each class is further split into **inter-chain** and **intra-chain**. Intra-chain contacts are mostly covalent neighbours (X with the Gly next to it), so they are shown separately and left out of the fractions.

**Input:** the existing `plf_files/{RES}_plf.csv` (ProLIF boolean fingerprints; ligand = every X residue of that type, protein = all residues). Nothing is re-run. Each CSV is reduced once to column sums and cached in `gly_sc_cache/`.

**Units:** a *contact* is one (ligand residue, partner residue, interaction type) that is `True` in a frame; `Any` is a residue pair with at least one interaction type `True`. Rates are **per X residue per frame**.

Caveats:
- ProLIF residue labels here are 0-based residue indices, so `chain = index // chain_length`.
- In the `GLY_plf.csv` files the ligand GLY is not the central residue (position 5 instead of 4 in the 9-res systems, the C-terminal Gly in the 3-res systems). Read the GLY row with that in mind.
- A Gly-partner contact means the *partner* is backbone-only; ProLIF residue-level output doesn't say whether the X side contributed backbone or side-chain atoms.

In [ ]:
import glob
import os
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import MDAnalysis as mda
from joblib import Parallel, delayed

import warnings
warnings.filterwarnings('ignore')

In [ ]:
BASE = '/Volumes/lss/research/potoyan-lab/will/gxg-project'
CACHE = f'{BASE}/notebooks/gly_sc_cache'
FIGS = f'{BASE}/notebooks/figures/gly_vs_sc'
os.makedirs(CACHE, exist_ok=True)
os.makedirs(FIGS, exist_ok=True)

# label: (directory, chain length)
SYSTEMS = {
    '9res_r1':  ('chain_length/9_res/rep_1', 9),
    '9res_r2':  ('chain_length/9_res/rep_2', 9),
    '15res_r1': ('chain_length/15_res/rep_1', 15),
    '15res_r2': ('chain_length/15_res/rep_2', 15),
    '21res_r1': ('chain_length/21_res/rep_1', 21),
    '21res_r2': ('chain_length/21_res/rep_2', 21),
    '3E': ('gxg_comps/3_res/equal', 3),
    '3C': ('gxg_comps/3_res/charged', 3),
    '3A': ('gxg_comps/3_res/aromatic', 3),
    '3P': ('gxg_comps/3_res/polar', 3),
    '3N': ('gxg_comps/3_res/nonpolar', 3),
    '9E': ('gxg_comps/9_res/old_equal', 9),
    '9C': ('gxg_comps/9_res/charged', 9),
    '9A': ('gxg_comps/9_res/aromatic', 9),
    '9P': ('gxg_comps/9_res/polar', 9),
    '9N': ('gxg_comps/9_res/nonpolar', 9),
}

N_JOBS = 4  # parallel CSV reads on the first (uncached) pass

res_order = ['ASP','GLU','ARG','LYS','HIS','TRP','TYR','PHE','ASN','GLN',
             'SER','THR','CYS','PRO','VAL','MET','ILE','LEU','ALA','GLY']

aromatic = ['PHE', 'TYR', 'TRP']
polar = ['CYS','THR','GLN','ASN','SER']
nonpolar = ['VAL','ILE','ALA','LEU','PRO','GLY','MET']
positive = ['ARG','LYS','HIS']
negative = ['GLU','ASP']

def res_color(res):
    if res in aromatic:
        return '#e3ba14'
    elif res in polar:
        return '#7aaa3d'
    elif res in positive:
        return '#b30039'
    elif res in negative:
        return '#0c7ace'
    return '#ac5cad'

def color_ticks(ax, axis='x'):
    labels = ax.get_xticklabels() if axis == 'x' else ax.get_yticklabels()
    for tick_label in labels:
        tick_label.set_color(res_color(tick_label.get_text()))

# ProLIF types merged the same way as in Prolif_Analysis.ipynb
itype_groups = {'HBDonor':'H-Bond', 'HBAcceptor':'H-Bond',
                'Cationic':'Electrostatic', 'Anionic':'Electrostatic',
                'CationPi':'Pi-Cation', 'PiCation':'Pi-Cation',
                'PiStacking':'Pi-Stacking', 'EdgeToFace':'Pi-Stacking', 'FaceToFace':'Pi-Stacking',
                'Hydrophobic':'Hydrophobic', 'VdWContact':'VdWContact', 'Any':'Any'}

split_cols = ['Inter Gly', 'Inter Non-Gly', 'Intra Gly', 'Intra Non-Gly']
partner_colors = dict(zip(split_cols, ['#7f7f7f', '#c0392b', '#d0d0d0', '#f1a9a0']))

## 1. Reduce the ProLIF CSVs (cached)

Each `{RES}_plf.csv` (~50–300 MB) is read in chunks and reduced to:
- the number of `True` frames for every (ligand, partner, interaction) column
- the number of frames where the (ligand, partner) pair has *any* interaction (`interaction = 'Any'`)

The first pass over all systems takes about an hour; after that the cache loads in seconds.

In [ ]:
def reduce_plf(path, chunksize=2000):
    n_frames = 0
    per_type = None
    per_pair = None
    for chunk in pd.read_csv(path, header=[0,1,2], index_col=0, chunksize=chunksize,
                             true_values=['True'], false_values=['False']):
        chunk = chunk.astype(bool)
        n_frames += len(chunk)
        s = chunk.sum()
        p = chunk.T.groupby(level=[0,1]).any().T.sum()
        per_type = s if per_type is None else per_type.add(s, fill_value=0)
        per_pair = p if per_pair is None else per_pair.add(p, fill_value=0)

    per_type = per_type.reset_index()
    per_type.columns = ['ligand','protein','interaction','count']
    per_pair = per_pair.reset_index()
    per_pair.columns = ['ligand','protein','count']
    per_pair['interaction'] = 'Any'

    df = pd.concat([per_type, per_pair], ignore_index=True)
    df['n_frames'] = n_frames
    return df


def cached_reduce(label, path):
    res = os.path.basename(path)[:3]
    cache_file = f'{CACHE}/{label}_{res}.csv'
    if os.path.exists(cache_file):
        return pd.read_csv(cache_file)
    df = reduce_plf(path)
    df.to_csv(cache_file, index=False)
    return df


jobs = []
for label, (sys_dir, L) in SYSTEMS.items():
    for path in sorted(glob.glob(f'{BASE}/{sys_dir}/plf_files/[A-Z][A-Z][A-Z]_plf.csv')):
        jobs.append((label, path))

print(len(jobs), 'files')
results = Parallel(n_jobs=N_JOBS, verbose=5)(delayed(cached_reduce)(label, path) for label, path in jobs)

raw = pd.concat([r.assign(system=label) for (label, _), r in zip(jobs, results)], ignore_index=True)
raw.head()

## 2. Annotate partner (Gly / non-Gly) and chain (intra / inter)

In [ ]:
LABEL = r'([A-Z]{3})(\d+)'

def annotate(df):
    df = df.copy()
    lig = df['ligand'].str.extract(LABEL)
    prot = df['protein'].str.extract(LABEL)
    df['lig_res'], df['lig_idx'] = lig[0], lig[1].astype(int)
    df['prot_res'], df['prot_idx'] = prot[0], prot[1].astype(int)

    df = df[df['lig_idx'] != df['prot_idx']]      # drop self "contacts"

    L = df['system'].map({k: v[1] for k, v in SYSTEMS.items()})
    df['chain'] = np.where(df['lig_idx'] // L == df['prot_idx'] // L, 'Intra', 'Inter')
    df['partner'] = np.where(df['prot_res'] == 'GLY', 'Gly', 'Non-Gly')
    df['itype'] = df['interaction'].map(itype_groups)

    # per X residue per frame
    n_lig = df.groupby(['system','lig_res'])['ligand'].transform('nunique')
    df['rate'] = df['count'] / (df['n_frames'] * n_lig)
    return df

conts = annotate(raw)

In [ ]:
# Sanity check: position of the ligand residue inside its chain (should be the centre, (L-1)//2)
lig_pos = conts.drop_duplicates(['system','ligand']).copy()
lig_pos['pos'] = lig_pos['lig_idx'] % lig_pos['system'].map({k: v[1] for k, v in SYSTEMS.items()})
lig_pos.groupby(['system','lig_res'])['pos'].unique().unstack('system').loc[res_order]

## 3. Contact split per residue

`split` is indexed by (system, residue). Columns are chain × partner contact rates per X residue per frame, computed from residue pairs (`Any`).

In [ ]:
def split_table(df, interaction='Any'):
    d = df[df['interaction'] == interaction]
    d = d.assign(cls=d['chain'] + ' ' + d['partner'])
    out = d.groupby(['system','lig_res','cls'])['rate'].sum().unstack('cls')
    return out.reindex(columns=split_cols).fillna(0)

split = split_table(conts)

split['Total (inter)'] = split['Inter Gly'] + split['Inter Non-Gly']
split['% Non-Gly (inter)'] = 100 * split['Inter Non-Gly'] / split['Total (inter)']
split.to_csv(f'{FIGS}/gly_vs_nongly_split.csv')
split.loc['9res_r1'].loc[res_order].round(3)

In [ ]:
def plot_split(system, interaction='Any', ax=None):
    d = split_table(conts[conts['system'] == system], interaction).droplevel('system')
    d = d.loc[(d['Inter Gly'] + d['Inter Non-Gly']).sort_values().index]

    if ax is None:
        fig, ax = plt.subplots(figsize=[9,5])
    d.plot(kind='bar', stacked=True, width=0.8, ax=ax,
           color=[partner_colors[c] for c in split_cols], edgecolor='none')
    ax.legend([f'{c.split(" ", 1)[1]} ({c.split(" ")[0].lower()}-chain)' for c in split_cols], fontsize=11, frameon=False)
    ax.set_ylabel(f'{interaction} contacts per residue per frame', fontsize=13)
    ax.set_xlabel('')
    ax.set_title(system, fontsize=14)
    ax.tick_params(axis='x', rotation=45, labelsize=12)
    color_ticks(ax)
    return ax

ax = plot_split('9res_r1')
plt.tight_layout()
plt.savefig(f'{FIGS}/9res_r1_split.pdf', dpi=500, transparent=True)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=[20,9], sharey=True)
for ax, system in zip(axes.flat, ['3E','3C','3A','3P','3N','9E']):
    plot_split(system, ax=ax)
    if ax is not axes.flat[0]:
        ax.get_legend().remove()
plt.tight_layout()
plt.savefig(f'{FIGS}/composition_split.pdf', dpi=500, transparent=True)
plt.show()

## 4. Share of inter-chain contacts made with side-chain residues (non-Gly)

Heatmap of `% Non-Gly (inter)`: of all inter-chain contacts residue X makes, the percentage that are with other X residues rather than with Gly.

In [ ]:
pct = split['% Non-Gly (inter)'].unstack('system').reindex(res_order)[list(SYSTEMS)]

fig, ax = plt.subplots(figsize=[12,7])
sns.heatmap(pct, cmap='rocket_r', annot=True, fmt='.0f', annot_kws={'fontsize':8},
            cbar_kws={'label':'% inter-chain contacts with non-Gly'}, ax=ax)
ax.set_xlabel('')
ax.set_ylabel('')
ax.tick_params(axis='x', rotation=45)
color_ticks(ax, 'y')
plt.tight_layout()
plt.savefig(f'{FIGS}/pct_nongly_heatmap.pdf', dpi=500, transparent=True)
plt.show()

## 5. Correcting for partner availability

The raw split is mostly set by composition: in a `GGGGXGGGG` system 8/9 of potential partners are Gly. To get a per-partner propensity we divide by the number of available partners of each class on *other* chains (counted from `output-nw.pdb`):

$$
\text{enrichment}(X) = \frac{\text{rate}_{\text{inter, non-Gly}} / N_{\text{non-Gly}}}{\text{rate}_{\text{inter, Gly}} / N_{\text{Gly}}}
$$

&gt; 1 means a single side-chain-bearing residue is a better partner for X than a single glycine.

In [ ]:
def partner_availability(label):
    sys_dir, L = SYSTEMS[label]
    u = mda.Universe(f'{BASE}/{sys_dir}/output-nw.pdb')
    resnames = u.select_atoms('protein').residues.resnames
    chain = np.arange(len(resnames)) // L
    is_gly = resnames == 'GLY'

    ligs = conts.loc[conts['system'] == label, ['lig_res','lig_idx']].drop_duplicates()
    rows = []
    for _, r in ligs.iterrows():
        other = chain != r['lig_idx'] // L
        rows.append({'lig_res': r['lig_res'],
                     'N_Gly': (other & is_gly).sum(),
                     'N_Non-Gly': (other & ~is_gly).sum()})
    return pd.DataFrame(rows).groupby('lig_res').mean().assign(system=label)

avail = pd.concat([partner_availability(s) for s in SYSTEMS]).set_index('system', append=True).swaplevel()
avail.head()

In [ ]:
enrich = pd.DataFrame({
    'per Gly partner': split['Inter Gly'] / avail['N_Gly'],
    'per Non-Gly partner': split['Inter Non-Gly'] / avail['N_Non-Gly'],
})
enrich['enrichment'] = enrich['per Non-Gly partner'] / enrich['per Gly partner']
enrich.to_csv(f'{FIGS}/nongly_enrichment.csv')

e = enrich['enrichment'].unstack('system').reindex(res_order)[list(SYSTEMS)]

fig, ax = plt.subplots(figsize=[12,7])
sns.heatmap(np.log2(e), cmap='vlag', center=0, annot=e, fmt='.1f', annot_kws={'fontsize':8},
            cbar_kws={'label':'log$_2$ enrichment (non-Gly / Gly, per partner)'}, ax=ax)
ax.set_xlabel('')
ax.set_ylabel('')
ax.tick_params(axis='x', rotation=45)
color_ticks(ax, 'y')
plt.tight_layout()
plt.savefig(f'{FIGS}/nongly_enrichment_heatmap.pdf', dpi=500, transparent=True)
plt.show()

## 6. Which interaction types go through Gly vs. through side chains?

Same split, now by ProLIF interaction type (merged: H-Bond = donor + acceptor, Electrostatic = cationic + anionic, Pi-Stacking = face/edge/stacking, Pi-Cation = both directions). Inter-chain only.

In [ ]:
inter = conts[(conts['chain'] == 'Inter') & (conts['interaction'] != 'Any')]

by_type = (inter.groupby(['system','lig_res','itype','partner'])['rate'].sum()
                .unstack('partner').fillna(0))
by_type['% Non-Gly'] = 100 * by_type['Non-Gly'] / (by_type['Gly'] + by_type['Non-Gly'])
by_type.to_csv(f'{FIGS}/gly_vs_nongly_by_itype.csv')

system = '9res_r1'
d = by_type.loc[system]

fig, axes = plt.subplots(1, 2, figsize=[16,7], gridspec_kw={'width_ratios':[1.3,1]})

itypes = ['VdWContact','Hydrophobic','H-Bond','Electrostatic','Pi-Cation','Pi-Stacking']
heat = d['% Non-Gly'].unstack('itype').reindex(index=res_order, columns=itypes)
sns.heatmap(heat, cmap='rocket_r', vmin=0, vmax=100, annot=True, fmt='.0f',
            cbar_kws={'label':'% inter-chain contacts with non-Gly'}, ax=axes[0])
axes[0].set_title(f'{system}: share via side-chain residues', fontsize=13)
axes[0].set_xlabel('')
axes[0].set_ylabel('')
color_ticks(axes[0], 'y')

tot = d[['Gly','Non-Gly']].groupby(level='itype').sum().reindex(itypes)
tot.plot(kind='barh', stacked=True, color=['#7f7f7f','#c0392b'], ax=axes[1], edgecolor='none')
axes[1].set_xlabel('Contacts per residue per frame (summed over residues)', fontsize=12)
axes[1].set_ylabel('')
axes[1].set_title('Absolute split by type', fontsize=13)
axes[1].legend(frameon=False)

plt.tight_layout()
plt.savefig(f'{FIGS}/{system}_itype_split.pdf', dpi=500, transparent=True)
plt.show()

## 7. Side-chain–side-chain contact map (non-Gly ↔ non-Gly, inter-chain)

Rate per X residue per frame of contacts between residue X (row) and partner residue Y (column), excluding Gly partners.

In [ ]:
def xx_matrix(system, interaction='Any'):
    d = conts[(conts['system'] == system) & (conts['interaction'] == interaction) &
              (conts['chain'] == 'Inter') & (conts['partner'] == 'Non-Gly')]
    m = d.groupby(['lig_res','prot_res'])['rate'].sum().unstack().fillna(0)
    return m.reindex(index=res_order, columns=res_order[:-1]).fillna(0)

system = '3E'
m = xx_matrix(system)

fig, ax = plt.subplots(figsize=[9,8])
sns.heatmap(m, cmap='YlOrBr', square=True, cbar_kws={'label':'contacts per residue per frame', 'shrink':0.7}, ax=ax)
ax.set_xlabel('Partner residue (non-Gly)', fontsize=13)
ax.set_ylabel('Residue X', fontsize=13)
ax.set_title(f'{system}: side-chain residue contacts', fontsize=14)
color_ticks(ax, 'x')
color_ticks(ax, 'y')
plt.tight_layout()
plt.savefig(f'{FIGS}/{system}_xx_matrix.pdf', dpi=500, transparent=True)
plt.show()

## 8. Chain-length dependence (9 → 15 → 21 residues)

Mean of the two replicas; error bars span the two replicas.

In [ ]:
lengths = {9: ['9res_r1','9res_r2'], 15: ['15res_r1','15res_r2'], 21: ['21res_r1','21res_r2']}
markers = {9:'o', 15:'s', 21:'D'}

fig, axes = plt.subplots(2, 1, figsize=[11,8], sharex=True)
x = np.arange(len(res_order))

for k, (n, reps) in enumerate(lengths.items()):
    for ax, (col, ylabel) in zip(axes, [('% Non-Gly (inter)', '% inter-chain contacts\nwith non-Gly'),
                                        ('Total (inter)', 'Inter-chain contacts\nper residue per frame')]):
        vals = pd.concat([split.loc[r][col] for r in reps], axis=1).reindex(res_order)
        mean = vals.mean(axis=1)
        err = [mean - vals.min(axis=1), vals.max(axis=1) - mean]
        ax.errorbar(x + (k-1)*0.22, mean, yerr=err, fmt=markers[n], color='k', mfc='white' if k else 'k',
                    capsize=2, label=f'{n} residues')
        ax.set_ylabel(ylabel, fontsize=12)

axes[0].legend(frameon=False, ncol=3)
axes[1].set_xticks(x)
axes[1].set_xticklabels(res_order, rotation=45, fontsize=12)
color_ticks(axes[1])
plt.tight_layout()
plt.savefig(f'{FIGS}/chain_length_gly_vs_nongly.pdf', dpi=500, transparent=True)
plt.show()

## 9. Summary by residue class

In [ ]:
res_class = {r: c for c, rs in {'Positive':positive, 'Negative':negative, 'Aromatic':aromatic,
                                'Polar':polar, 'Nonpolar':nonpolar}.items() for r in rs}

summary = split[['Inter Gly', 'Inter Non-Gly', '% Non-Gly (inter)']].copy()
summary.columns = ['Inter Gly', 'Inter Non-Gly', '% Non-Gly']
summary['enrichment'] = enrich['enrichment']
summary = summary.reset_index()
summary = summary[summary['lig_res'] != 'GLY']
summary['class'] = summary['lig_res'].map(res_class)

summary.groupby(['system','class'])[['Inter Gly','Inter Non-Gly','% Non-Gly','enrichment']].mean().unstack('class').round(2)